# ARC AGI 3 Colab Training

This notebook does five things:

1. Copies the project from Google Drive to the Colab local disk
2. Installs the required dependencies
3. Optionally converts the human replay zip into training ready `episodes.jsonl.gz`
4. Optionally runs public environment collection
5. Trains from one or more `.gz` trajectory files and writes logs, validation outputs, and checkpoints to `ARC Prize 2026_AGI_3/Training_Output/<timestamp>/`

Project source can come from either a synced Google Drive folder or a single project zip bundle stored on Drive. Keep the notebook thin and keep real code in files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

PROJECT_SOURCE_MODE = 'drive_dir'  # Use 'drive_zip' if you uploaded a project bundle zip instead of a full Drive folder.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3')
DRIVE_PROJECT_ZIP = Path('/content/drive/MyDrive/arc_agi3_colab_bundle.zip')
DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC Prize 2026_AGI_3/Training_Output')
DRIVE_COLLECTION_BASE = Path('/content/drive/MyDrive/ARC Prize 2026_AGI_3/Collection_Cache')
LOCAL_WORKDIR = Path('/content/ARC Prize 2026 - ARC-AGI-3')
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_COLLECTION_BASE.mkdir(parents=True, exist_ok=True)

print('Project source mode:', PROJECT_SOURCE_MODE)
print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Drive project zip:', DRIVE_PROJECT_ZIP)
print('Drive output base:', DRIVE_OUTPUT_BASE)
print('Drive collection base:', DRIVE_COLLECTION_BASE)
print('Local workdir:', LOCAL_WORKDIR)
print('Output root:', OUTPUT_ROOT)


In [ ]:
import shutil

if LOCAL_WORKDIR.exists():
    shutil.rmtree(LOCAL_WORKDIR)
LOCAL_WORKDIR.parent.mkdir(parents=True, exist_ok=True)

if PROJECT_SOURCE_MODE == 'drive_dir':
    get_ipython().system('rsync -a --delete --exclude .git "{}"/ "{}"/'.format(DRIVE_PROJECT_ROOT, LOCAL_WORKDIR))
elif PROJECT_SOURCE_MODE == 'drive_zip':
    if not DRIVE_PROJECT_ZIP.exists():
        raise FileNotFoundError(f'Project zip not found: {DRIVE_PROJECT_ZIP}')
    get_ipython().system('unzip -q "{}" -d /content'.format(DRIVE_PROJECT_ZIP))
else:
    raise ValueError(f'Unsupported PROJECT_SOURCE_MODE: {PROJECT_SOURCE_MODE}')

get_ipython().run_line_magic('cd', str(LOCAL_WORKDIR))


In [ ]:
import os, sys, subprocess

def run(cmd):
    print('>>>', cmd)
    subprocess.check_call(cmd, shell=True)

run('python -m pip install -U pip wheel setuptools')
run('python -m pip install -U torch torchvision torchaudio')

try:
    run('python -m pip install -U arc-agi==0.9.8 arcengine==0.9.3')
except Exception:
    print('PyPI install failed, trying local wheels...')
    run('python -m pip install arc_agi_3_wheels/*.whl')

run('python - <<\'PY\'\nimport torch\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("device", torch.cuda.get_device_name(0))\nPY')


In [ ]:
HARDWARE_PROFILE = 'a100'  # Training profile. Switch to 'h100' only if needed.
COLLECT_PROFILE = 'a100'  # For smoke tests use 'cpu_debug'.
COLLECT_TAG = 'public_search_a100_v1'
RUN_COLLECTION = False  # Set to True only when you want to build or refresh cached self collect trajectories.
RUN_HUMAN_IMPORT = False  # Set to True if you uploaded the official human replay zip to Drive and want to convert it here.
HUMAN_ZIP_PATH = Path('/content/drive/MyDrive/arc_agi_3_public_demo_human_testing.zip')
HUMAN_IMPORT_TAG = 'human_focus5_v1'
HUMAN_IMPORT_GAMES = 'sp80,lp85,ar25,ls20,r11l'
HUMAN_IMPORT_MIN_LEVELS = 1
HUMAN_IMPORT_TOP_K = 10
COLLECT_STEPS = 96
COLLECT_WORKERS = 8  # Use more only if the runtime has the CPU cores and RAM to support it.
COLLECT_GAMES = None  # Example: 'ls20,ar25'
COLLECT_EPISODES_PER_GAME = None  # Example: 8 for a cheaper first pass
COLLECT_BEAM_WIDTH = None  # Example: 4
COLLECT_BRANCH_FACTOR = None  # Example: 6
CHECKPOINT_EVERY_STEPS = 100
RESUME_CHECKPOINT = None  # Example: OUTPUT_ROOT / 'checkpoints' / 'last.pth'

COLLECT_ROOT = DRIVE_COLLECTION_BASE / COLLECT_TAG
HUMAN_IMPORT_ROOT = DRIVE_COLLECTION_BASE / HUMAN_IMPORT_TAG
HUMAN_IMPORT_OUTPUT = HUMAN_IMPORT_ROOT / 'collected' / 'episodes.jsonl.gz'
TRAIN_DATA_PATHS = []  # Example: [Path('/content/drive/MyDrive/ARC Prize 2026_AGI_3/Collection_Cache/human_focus5_v1/collected/episodes.jsonl.gz')]
collect_optional_args = []
collect_optional_args.append(f'--workers {COLLECT_WORKERS}')
if COLLECT_GAMES:
    collect_optional_args.append(f'--games {COLLECT_GAMES}')
if COLLECT_EPISODES_PER_GAME is not None:
    collect_optional_args.append(f'--episodes-per-game {COLLECT_EPISODES_PER_GAME}')
if COLLECT_BEAM_WIDTH is not None:
    collect_optional_args.append(f'--beam-width {COLLECT_BEAM_WIDTH}')
if COLLECT_BRANCH_FACTOR is not None:
    collect_optional_args.append(f'--branch-factor {COLLECT_BRANCH_FACTOR}')
collect_optional = '' if not collect_optional_args else ' \\\n  ' + ' \\\n  '.join(collect_optional_args)

if RUN_HUMAN_IMPORT:
    human_cmd = f'''python -m src.import_human_replays \
  --project-root "{LOCAL_WORKDIR}" \
  --input "{HUMAN_ZIP_PATH}" \
  --output "{HUMAN_IMPORT_OUTPUT}" \
  --games "{HUMAN_IMPORT_GAMES}" \
  --min-levels {HUMAN_IMPORT_MIN_LEVELS} \
  --top-k-per-game {HUMAN_IMPORT_TOP_K}'''
    run(human_cmd)
    TRAIN_DATA_PATHS.append(HUMAN_IMPORT_OUTPUT)

if RUN_COLLECTION:
    collect_cmd = f'''python -m src.collect \
  --project-root "{LOCAL_WORKDIR}" \
  --output-root "{COLLECT_ROOT}" \
  --hardware-profile {COLLECT_PROFILE} \
  --seeds 0,1,2,3 \
  --max-steps {COLLECT_STEPS}{collect_optional}'''
    run(collect_cmd)
    TRAIN_DATA_PATHS.append(COLLECT_ROOT / 'collected' / 'episodes.jsonl.gz')

if not TRAIN_DATA_PATHS:
    TRAIN_DATA_PATHS = [
        HUMAN_IMPORT_OUTPUT,
        COLLECT_ROOT / 'collected' / 'episodes.jsonl.gz',
    ]

TRAIN_DATA_PATHS = [path for path in TRAIN_DATA_PATHS if path.exists()]
if not TRAIN_DATA_PATHS:
    raise FileNotFoundError('No training data paths were found. Enable RUN_HUMAN_IMPORT or RUN_COLLECTION, or point the defaults at existing .gz files.')

TRAIN_DATA_ARG = ','.join(str(path) for path in TRAIN_DATA_PATHS)
print('Training data paths:')
for path in TRAIN_DATA_PATHS:
    print(' -', path)


In [ ]:
resume_arg = '' if RESUME_CHECKPOINT is None else f' \\\n  --resume "{RESUME_CHECKPOINT}"'

train_cmd = f'''python -m src.train \
  --project-root "{LOCAL_WORKDIR}" \
  --data "{TRAIN_DATA_ARG}" \
  --output-dir "{OUTPUT_ROOT}" \
  --hardware-profile {HARDWARE_PROFILE} \
  --max-steps 192 \
  --online-val-games 5 \
  --checkpoint-every-steps {CHECKPOINT_EVERY_STEPS}{resume_arg}'''

run(train_cmd)


In [ ]:
eval_cmd = f'''python -m src.evaluate \
  --project-root "{LOCAL_WORKDIR}" \
  --checkpoint "{OUTPUT_ROOT / 'checkpoints' / 'best.pth'}" \
  --output "{OUTPUT_ROOT / 'public_eval.json'}" \
  --split val'''

run(eval_cmd)


In [ ]:
import pandas as pd
from pathlib import Path

metrics_path = OUTPUT_ROOT / 'metrics.csv'
display(pd.read_csv(metrics_path).tail())
print('Best checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'best.pth')
print('Public eval:', OUTPUT_ROOT / 'public_eval.json')
